# Assignment 6: Smoothing Filters


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

gray_img = cv2.imread('Sunflower_from_Silesia2.jpg', cv2.IMREAD_GRAYSCALE)


## Filter Parameters & Padding


In [ ]:
try:
    k_in = input('Enter filter size (e.g. 3, 5, 7) [Default 5]: ')
    sigma_in = input('Enter sigma for Gaussian filter [Default 1.2]: ')
    pad_in = input('Enter padding method (zero / replicate / reflect) [Default replicate]: ')

    ksize = int(k_in) if k_in.strip() != '' else 5
    if ksize % 2 == 0:
        ksize += 1
    sigma = float(sigma_in) if sigma_in.strip() != '' else 1.2
    pad_method = pad_in.strip().lower() if pad_in.strip() != '' else 'replicate'
    if pad_method not in ['zero', 'replicate', 'reflect']:
        pad_method = 'replicate'
except (ValueError, EOFError):
    ksize = 5
    sigma = 1.2
    pad_method = 'replicate'


In [ ]:
def apply_spatial_filter(image, kernel, padding='replicate'):
    h, w = image.shape
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2

    if padding == 'zero':
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)
    elif padding == 'replicate':
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')
    elif padding == 'reflect':
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    else:
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')

    output = np.zeros((h, w), dtype=np.float64)
    padded_float = padded.astype(np.float64)

    for i in range(h):
        for j in range(w):
            region = padded_float[i:i + kh, j:j + kw]
            output[i, j] = np.sum(region * kernel)

    return np.uint8(np.clip(np.round(output), 0, 255))


## Smoothing Filters: Averaging, Weighted, and Gaussian


In [ ]:
radius = ksize // 2
coords = np.arange(-radius, radius + 1)
x, y = np.meshgrid(coords, coords)

kernel_avg = np.ones((ksize, ksize), dtype=np.float64) / (ksize * ksize)

kernel_weighted = np.maximum(np.abs(x), np.abs(y)).astype(np.float64)
if np.sum(kernel_weighted) > 0:
    kernel_weighted /= np.sum(kernel_weighted)
else:
    kernel_weighted = kernel_avg

kernel_gauss = np.exp(-(x**2 + y**2) / (2.0 * sigma**2))
kernel_gauss /= np.sum(kernel_gauss)

smooth_avg = apply_spatial_filter(gray_img, kernel_avg, padding=pad_method)
smooth_weighted = apply_spatial_filter(gray_img, kernel_weighted, padding=pad_method)
smooth_gauss = apply_spatial_filter(gray_img, kernel_gauss, padding=pad_method)

cv_gauss = cv2.GaussianBlur(gray_img, (ksize, ksize), sigmaX=sigma, borderType=cv2.BORDER_REPLICATE)

plt.figure(figsize=(16, 8))

plt.subplot(2, 3, 1)
plt.title('Original Image')
plt.imshow(gray_img, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 2)
plt.title(f'1. Averaging Filter ({ksize}x{ksize})')
plt.imshow(smooth_avg, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 3)
plt.title('2. Weighted Filter (max(|x|,|y|))')
plt.imshow(smooth_weighted, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 4)
plt.title(f'3. Gaussian Filter (\u03c3={sigma})')
plt.imshow(smooth_gauss, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 5)
plt.title('OpenCV GaussianBlur')
plt.imshow(cv_gauss, cmap='gray')
plt.axis('off')

plt.subplot(2, 3, 6)
plt.title('Difference (Manual vs OpenCV)')
diff = np.abs(smooth_gauss.astype(int) - cv_gauss.astype(int))
plt.imshow(diff, cmap='hot')
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis('off')

plt.tight_layout()
plt.show()
